In [ ]:
fig,ax = plt.subplots(1, 5, sharey=True)
fig.set_figwidth(10)
fig.set_figheight(2.25)
fig.tight_layout()
# plots per categories
for i, category in enumerate(categories):
    mean_power.drop(mean_power[mean_power["category"] != category].index).plot(kind="bar", ax=ax[i], title=category.capitalize())
    handles, labels = ax[i].get_legend_handles_labels()
    ax[i].get_legend().remove()
    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.7))
ax[0].set_ylabel("Mean Power (W)")
# plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
# fig.savefig('assets/all.png', bbox_inches="tight")

In [ ]:
# active.csv
# idle.csv

# Load data

columns = ['active', 'idle']
data = {'category': [category for category, xs in devices.items() for x in xs]}
raw = {}
for column in columns:
    power = []
    for category, ds in devices.items():
        for d in ds:
            df = pd.read_csv(f'{BASE_PATH}/{category}/{d}/{column}.csv')
            power.append(df.mean(numeric_only=True).to_list()[0] / 1000)
    data[column] = power

devices_tuples = [(d, category) for category, ds in devices.items() for d in ds]
mean_power = pd.DataFrame(index=devices_flat, data=data)
num_devices = len(devices_flat)
print(mean_power)
print(devices_tuples)
print(len(devices_flat))

mean_power = mean_power.drop('cosori')
mean_power = mean_power.drop('samsungtv')

In [ ]:
plt.figure(figsize=(28,8))
diff = (mean_power['idle'] - mean_power['active']) / mean_power['active'] * 100
diff.plot(kind="bar")
plt.ylabel("% Reduction");
plt.rcParams.update({'font.size': 32})
min_y = -110
max_y = 11
plt.title("Power Change Active to Idle")

plt.ylim(min_y, max_y);
plt.yticks(range(min_y + 10, max_y, 10))
# plt.savefig("assets/diff.png", bbox_inches="tight")


# d = get_dataframe("audio", "googlehome", "active")
# d['time'] = np.linspace(0, 50*60, 2313)
plot_devices = num_devices
# plot_devices = 2
print(num_devices)
cols = 3
fig, ax = plt.subplots(plot_devices // cols, cols)
fig.set_figheight(2 * plot_devices)
fig.set_figwidth(30)
fig.tight_layout(pad=3)

# plot("googlehome", "audio", "active", ax[0], lambda x: x)
# plot("echodot3", "audio", "active", ax[1], lambda x: x)


# plt.errorbar
# np.std


for i, device in enumerate(devices_flat[:plot_devices]):
    # a = plot(device, category, "active", ax[i], lambda x: x[::50])
    axis = ax[i // cols, i % cols]
    plot(device, "active", axis, lambda x: x)

    # ax[i, 0].set_ylim(0, max(a, b) * 1.1)
    # ax[i, 1].set_ylim(0, max(a, b) * 1.1)
    # axis.set_xticks(np.arange(0, 50, 2))
    axis.set_title(f'{device} ({category})')
    axis.set_ylabel("Power (W)")
    axis.set_xlabel("Time (minutes)")
    # ax[i].set_title(f'{device} ({category}) zoomed')
# plt.savefig("report/assets/time.png", bbox_inches="tight")
plt.show()

In [ ]:
#| label: plot1
times = []
means = []
stds = []
samples = []
fig,ax = plt.subplots(1,5)

categories_labels = []
for i, (device, (df, _)) in enumerate(get_data("idle").items()):
    category = devices_to_cat[device]
    if category in categories_labels:
        category_label = f'_{category}'
    else:
        category_label = category
    categories_labels.append(category)
    colour = ColorHash(devices_to_cat[device]).hex

    mean = df.power_mW.mean()
    std = df.power_mW.std()
    times.append(df.timestamp.iat[-1] - df.timestamp[0])
    means.append(mean)
    stds.append(std)
    samples.append(len(df))
   
    ax[categories.index(category)].errorbar(device, mean, std, linestyle='none', markersize=15, markerfacecolor=colour, markeredgecolor='none', marker=".", label=category_label)


width = 15
ratio = 0.2
fig.set_figwidth(width)
fig.set_figheight(width * ratio)


means = np.array(means)
stds = np.array(stds)
for i in range(5):
    ax[i].tick_params('x', rotation=90)
    ax[i].legend()
plt.show()

def my_style(styler):
    styler.format(lambda v: f'{v.total_seconds() / 60:.0f} m {v.total_seconds() % 60:.0f} s', 'minutes')
    styler.format(lambda v: f'{v:.0f} mW', subset='means')
    styler.format(lambda v: f'{v:.0f}', subset='std')
    styler.format(lambda v: f'{v * 100:.0f}%', subset='cv')
    styler.relabel_index(["Minutes", "samples", "μ", "σ", "CV"], axis="columns")
    return styler

results = pd.DataFrame(index=devices_flat, data={ "minutes": times, 'samples': samples, 'means': means, 'std': stds, 'cv': stds/means})
results.style.pipe(my_style)

In [ ]:
# | label: profiles

def graph_profiles(mode, samples, offset, metric='mean', zoom=None):
    # get the profiles for the metric, sorted size of spread
    data = get_data(mode)
    results = {device: compute_profile(df, samples, offset)[
        metric] for device, df in data.items()}
    # print(results)

    fig_numberline, numberline = plt.subplots(1, 1)
    numberline.set_title(
        f"Windowed Data {mode}.  s = {samples}, o = {offset} metric: {metric} {'zoomed to ' + str(zoom) if zoom else ''}")

    points = []

    # get the highest value so we know the xlim
    if not zoom:
        xlim = (0, max(results.values(), key=lambda x: x[-1])[-1])
    else:
        xlim = zoom
    for height, (device, result) in enumerate(results.items()):
        dev_colour = ColorHash(device).hex

        # do all the plotting manually, using annotate and scatter
        points.append(numberline.scatter(
            result, [height] * len(result), c=dev_colour, marker="."))

    numberline.tick_params('x', rotation=90)

    numberline.set_xticks(np.linspace(0, round(xlim[1], -3), 36))
    numberline.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, pos: f"{int(round(v, -2))}"))

    numberline.set_yticks(range(num_devices), labels=devices_flat)
    numberline.tick_params(axis='y', which='major', labelsize=8)
    numberline.set_ylabel("Device")
    if metric == 'mean':
        numberline.set_xlabel("Power (mW)")

    numberline.grid(axis='y', color="#0000004B", linestyle='-', linewidth=0.5)
    # fig_numberline.set_figheight(1)
    fig_numberline.set_figwidth(14)

    numberline.set_xlim(xlim)

    # fig_numberline.draw_without_rendering()
    # for point in points:
    #     bbox = point.get_tightbbox()
    #     box = patches.Rectangle(
    #         (bbox.x0, bbox.y0), bbox.x1 - bbox.x0, bbox.y1 - bbox.y0)
    #     box.set_color("black")
    #     box.set_fill(True)
    #     numberline.add_patch(box)
 
    # TODO readd legend


# exclude=['cosori', 'samsungtv', 'xiaomiblender', 'swan']
graph_profiles("active", 400, 10)
graph_profiles("active", 400, 10, zoom=(0, 5000))
graph_profiles("active", 400, 10, metric='std')
plt.show()

In [ ]:
offsets = np.arange(0, 1001, 50)[1:]
lengths = np.arange(60, 601, 30)

z = np.zeros((len(offsets), len(lengths)))
data = get_data(State.IDLE)
for i, offset in enumerate(offsets):
    for j, length in enumerate(lengths):
        z[i, j] = compute_overlap(data, length, offset, metrics=['std', 'mean']) / num_devices

In [ ]:
def plot(device, state: State, ax, function):
    df = get_dataframe(device, state)
    # every tenth value to speedup    
    df['time'] = np.linspace(0, 50, len(df))
    time = function(df['time'])
    
    power = function(df['power_mW']) / 1000

    ax.plot(time, power)
    return power.max()